# 05 · Test inference and submission

1. Stage-1 and stage-2 scores for every test candidate (average of the 5 fold models).
2. Frozen decision rule from `decision.json`.
3. Invariants asserted before writing: predicted ⊆ candidates, no record under two S1s, only known S1 ids; every test S1 gets exactly one row (empty lists allowed).
4. Official validator.
5. Submission zip in the required layout.

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Override settings here or with EF_* environment variables before starting Jupyter.
# os.environ["EF_DEV_MODE"] = "1"     # small consistent slice: end-to-end smoke test on a laptop
# os.environ["EF_N_THREADS"] = "32"

import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(f"root={S.root}\nwork_dir={S.work_dir}\ndev_mode={S.dev_mode} threads={S.n_threads}")

In [ ]:
scores = stages.run_predict_test(S)
scores.group_by("ckey").agg(pl.len(), pl.col("p").mean())

In [ ]:
summary = stages.run_write_submission(S)
summary

## Sanity: predicted set sizes per country (France is unseen in training)

In [ ]:
from entity_forge.io import read_tsv, MATCH_HEADER
res = read_tsv(S.output_dir / "matching_results.tsv", MATCH_HEADER).with_columns(
    pl.col("matched_entity_ids").fill_null("").str.split(",").list.eval(pl.element().filter(pl.element() != "")).list.len().alias("n"))
s1c = pl.read_parquet(S.norm("test", 1), columns=["entity_id", "ckey"])
(res.join(s1c, left_on="source1_entity_id", right_on="entity_id")
    .group_by("ckey").agg((pl.col("n") == 0).mean().alias("empty_rate"), pl.col("n").mean().alias("avg_matches"), pl.len()))

Training reference: empty rate ≈ 5.6 %, average ≈ 3.46 matches per S1. Large deviations for France suggest a normalization gap.

In [ ]:
print(stages.run_validator(S))

In [ ]:
zip_path = stages.build_submission_zip(S)
zip_path